# Final Benchmark Analysis — Dengue NS1 β-Ladder Binder Campaign

This notebook walks through the reproducible pieces of this project's benchmark:

1. Load the validated Boltz-2 scoring harness output (or re-run it on new sequences).
2. Score epitope fidelity with the clash-floor-corrected contact analysis.
3. Apply the locked acceptance gate and build the contact-first ranked table.
4. Check DENV1–4 pan-serotype conservation, including the target-vs-reference check.
5. Reproduce the campaign summary figure.

All harness code lives in `../harness/`; all reference data in `../data/`;
final results in `../results/`.


## 1. Setup

In [ ]:
import sys, json, csv
sys.path.insert(0, "../harness")
import epitope_contact_analysis as eca
import denv_conservation_check as dcc

import matplotlib.pyplot as plt
import numpy as np


## 2. Load Boltz-2 validation results

This project ran two independent rounds of Boltz-2 validation (Phase 2: direct
seed-5 refinement, 16 designs; Phase 2b: re-seeded from the one on-epitope
design, 16 designs). Load the saved per-design metrics tables from `../results/`.
If you're validating NEW sequences, run `../harness/boltz2_validation_harness.sh`
first and point this notebook at its `results/` directory instead.


In [ ]:
boltz_phase2 = list(csv.DictReader(open("../results/boltz_shortlist_results.csv")))
boltz_phase2b = list(csv.DictReader(open("../results/boltz_1b_results.csv")))
print(f"Phase 2 (direct refinement): {len(boltz_phase2)} designs")
print(f"Phase 2b (on-epitope re-seed): {len(boltz_phase2b)} designs")


## 3. Epitope fidelity + locked gate

Re-run the clash-floor-corrected contact analysis directly on the Boltz-2
predicted structures (do not trust an upstream tool's own contact count without
re-deriving it independently — see `epitope_contact_analysis.py`'s docstring for
why). Then apply the locked acceptance gate.


In [ ]:
BETA_LADDER = range(261, 306)
HOTSPOTS = [269, 272, 274, 294, 296]

def score_design(pdb_path, iptm, ptm, complex_plddt, complex_ipde):
    contact = eca.analyze_complex(pdb_path, binder_chain="A", target_chain="B",
                                    epitope_range=BETA_LADDER, hotspots=HOTSPOTS,
                                    residue_offset=0)  # Boltz-2 output uses native numbering
    passes = eca.passes_locked_gate(complex_plddt, iptm, complex_ipde, contact["n_epitope_contacts"])
    return {**contact, "iptm": iptm, "ptm": ptm, "complex_plddt": complex_plddt,
            "complex_ipde": complex_ipde, "passes_locked_gate": passes}

# Example (requires the predicted PDB files -- see boltz_shortlist_full_results.json / boltz_1b_full_results.json
# for the full per-design results already computed in this campaign):
print("See boltz_shortlist_full_results.json and boltz_1b_full_results.json in ../results/")
print("for the full pre-computed per-design results (structures were not re-bundled here for size).")


## 4. Contact-first ranking

Rank by: epitope contacts (desc) -> number of distinct hotspots engaged (desc) -> ipTM (desc).
This is the project's locked ranking rule -- it prioritizes physical epitope
engagement over the (demonstrated unreliable, for this scaffold) confidence score.


In [ ]:
final_table = list(csv.DictReader(open("../results/final_ranked_binder_table.csv")))
n_pass = sum(1 for r in final_table if r["passes_locked_gate"] == "True")
print(f"Total designs across both rounds: {len(final_table)}")
print(f"Designs passing the full locked gate: {n_pass}/{len(final_table)}")
print()
print(f"{'rank':4} {'batch':26} {'name':26} {'iptm':7} {'epi_ctc':8}")
for r in final_table[:10]:
    print(f"{r['overall_rank']:<4} {r['batch']:<26} {r['name']:<26} {r['boltz_iptm']:<7} {r['n_epitope_contacts']:<8}")


## 5. DENV1–4 pan-serotype check

Two checks: are the hotspots conserved among the four reference serotypes, AND
does the actual modeling target sequence match those references? (The second
check caught a real divergence in this campaign — see `denv_conservation_check.py`.)


In [ ]:
serotypes = dcc.load_fasta("../data/ns1_serotypes.fasta")
ref_report = dcc.check_reference_conservation(serotypes, HOTSPOTS)
for r in ref_report:
    print(r)


In [ ]:
# If you have your own modeling target sequence, check it against the references directly:
# target_report = dcc.check_target_vs_references(serotypes, YOUR_TARGET_SEQ, HOTSPOTS)
# for r in target_report:
#     print(r)
#
# This campaign's own target sequence diverged from all 4 references at position 272
# (target=R, all refs=K) -- see denv1-4_betaladder_conservation.csv in ../results/ for
# the full corrected table.
conservation_table = list(csv.DictReader(open("../results/denv1-4_betaladder_conservation.csv")))
hotspot_rows = [r for r in conservation_table if r["is_hotspot"] == "True"]
for r in hotspot_rows:
    print(r["position"], "used_target:", r["used_campaign_target_residue"],
          "matches_all_refs:", r["used_target_matches_all_DENV1-4_refs"])


## 6. Campaign summary figure

Reproduce the key figure: Boltz-2 ipTM vs. β-ladder epitope contacts, both
refinement rounds. The pattern shown here — high ipTM and on-epitope binding
essentially never co-occur — is the campaign's central finding.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
markers = {"Phase1_seed5_direct": "o", "Phase1b_onepitope_reseed": "^"}
for batch, marker in markers.items():
    subset = [r for r in final_table if r["batch"] == batch]
    iptms = [float(r["boltz_iptm"]) for r in subset]
    epi = [int(r["n_epitope_contacts"]) for r in subset]
    colors = ["#1b7837" if e >= 3 else "#b2182b" for e in epi]
    ax.scatter(iptms, epi, c=colors, marker=marker, s=60, edgecolor="black", lw=0.5)
ax.axvline(0.7, color="grey", ls="--", lw=1)
ax.axhline(3, color="grey", ls=":", lw=1)
ax.set_xlabel("Boltz-2 ipTM")
ax.set_ylabel("β-ladder epitope contacts")
ax.set_title("0/32 designs both clear ipTM>0.7 AND bind on-epitope")
plt.tight_layout()
plt.show()


## Summary

- **0/32** Boltz-2-validated designs across two independent refinement rounds pass
  the full locked acceptance gate.
- The ipTM/epitope-fidelity inversion replicated **4 independent times** across this
  project (5-seed analysis, 6-seed robustness check, Phase 2, Phase 2b).
- **4/5** designed hotspots are both cross-serotype-conserved and correctly
  represented in the actual modeling target sequence (position 272 diverges).
- See `../seed5_refinement_methods.md` and `../README.md` for full narrative detail,
  including disclosed data-integrity corrections made during this campaign.
